# X-Hour 2: Text Classification Workshop

**CS 165: Natural Language Processing**  
**Week 2 - Thursday X-Hour**

---

## Learning Objectives

By the end of this session, you will:
1. Build text classifiers from scratch using scikit-learn
2. Understand different text representation methods (BoW, TF-IDF, embeddings)
3. Compare Naive Bayes, Logistic Regression, and Neural approaches
4. Evaluate classifier performance using appropriate metrics
5. Debug common issues in text classification pipelines

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q scikit-learn pandas numpy matplotlib seaborn torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All imports successful!")

## Part 1: Loading Real Data

We'll use the **20 Newsgroups** dataset - a classic text classification benchmark containing posts from 20 different newsgroups. For this workshop, we'll focus on a subset to keep things manageable.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Select 4 categories that are relatively distinct
categories = [
    'sci.space',           # Science discussions about space
    'rec.sport.hockey',    # Sports discussions about hockey  
    'talk.politics.misc',  # Political discussions
    'comp.graphics'        # Computer graphics
]

# Load training data
print("Loading 20 Newsgroups dataset...")
train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')  # Remove metadata
)

# Load test data
test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

print(f"\n✓ Loaded {len(train_data.data)} training documents")
print(f"✓ Loaded {len(test_data.data)} test documents")
print(f"\nCategories: {train_data.target_names}")

### Explore the Data

In [ ]:
# Look at class distribution
train_labels = [train_data.target_names[i] for i in train_data.target]
label_counts = Counter(train_labels)

plt.figure(figsize=(10, 5))
plt.bar(label_counts.keys(), label_counts.values())
plt.xlabel('Category')
plt.ylabel('Number of Documents')
plt.title('Training Data Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nClass distribution:")
for label, count in label_counts.items():
    print(f"  {label}: {count} documents")

In [ ]:
# Examine sample documents
def show_examples(data, n=2):
    """Show example documents from each category."""
    for category_id in range(len(data.target_names)):
        category = data.target_names[category_id]
        print(f"\n{'='*80}")
        print(f"Category: {category}")
        print(f"{'='*80}")
        
        # Find documents in this category
        category_docs = [i for i, target in enumerate(data.target) if target == category_id]
        
        # Show first n
        for i, doc_id in enumerate(category_docs[:n], 1):
            text = data.data[doc_id]
            # Show first 300 characters
            preview = text[:300].replace('\n', ' ')
            print(f"\nExample {i}:")
            print(f"{preview}...")

show_examples(train_data, n=1)

### 💡 Exercise 1: Data Exploration

1. What patterns do you notice in the different categories?
2. What words/phrases might be good indicators of each category?
3. Are there any categories that might be hard to distinguish? Why?

**Your answers here:**

1. Patterns: 
2. Indicator words:
3. Difficult distinctions:

---

## Part 2: Feature Engineering for Text

Before we can classify text, we need to convert it to numerical features. Let's explore different approaches.

### Method 1: Bag of Words (BoW)

The simplest approach: count how many times each word appears.

In [ ]:
# Create a simple Bag of Words vectorizer
bow_vectorizer = CountVectorizer(
    max_features=5000,      # Keep only top 5000 words
    min_df=2,               # Word must appear in at least 2 documents
    max_df=0.8,             # Word must appear in less than 80% of documents
    stop_words='english'    # Remove common words
)

# Fit on training data and transform
X_train_bow = bow_vectorizer.fit_transform(train_data.data)
X_test_bow = bow_vectorizer.transform(test_data.data)

print(f"BoW vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"Training matrix shape: {X_train_bow.shape}")
print(f"Matrix sparsity: {(1 - X_train_bow.nnz / np.prod(X_train_bow.shape)) * 100:.1f}%")

# Show some features
feature_names = bow_vectorizer.get_feature_names_out()
print(f"\nExample features: {feature_names[:20]}")

In [ ]:
# Visualize a single document's BoW representation
doc_idx = 0
doc_bow = X_train_bow[doc_idx].toarray()[0]

# Get non-zero features
nonzero_indices = np.nonzero(doc_bow)[0]
nonzero_features = [(feature_names[i], doc_bow[i]) for i in nonzero_indices]
nonzero_features.sort(key=lambda x: x[1], reverse=True)

print(f"Document: {train_data.data[doc_idx][:200]}...\n")
print(f"Category: {train_data.target_names[train_data.target[doc_idx]]}\n")
print("Top 15 words by count:")
for word, count in nonzero_features[:15]:
    print(f"  {word}: {int(count)}")

### Method 2: TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF downweights common words and upweights rare, informative words.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

where:
- $\text{TF}(t, d)$ = frequency of term $t$ in document $d$
- $\text{IDF}(t) = \log\frac{N}{\text{df}(t)}$ = inverse document frequency of term $t$

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    stop_words='english',
    use_idf=True,
    sublinear_tf=True  # Use log scaling for term frequency
)

# Fit and transform
X_train_tfidf = tfidf_vectorizer.fit_transform(train_data.data)
X_test_tfidf = tfidf_vectorizer.transform(test_data.data)

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Training matrix shape: {X_train_tfidf.shape}")

In [ ]:
# Compare BoW vs TF-IDF for the same document
doc_idx = 0
doc_tfidf = X_train_tfidf[doc_idx].toarray()[0]
doc_bow = X_train_bow[doc_idx].toarray()[0]

feature_names = tfidf_vectorizer.get_feature_names_out()
nonzero_indices = np.nonzero(doc_tfidf)[0]

# Create comparison
comparison = [(feature_names[i], doc_bow[i], doc_tfidf[i]) for i in nonzero_indices]
comparison.sort(key=lambda x: x[2], reverse=True)

print("Top 15 words by TF-IDF score:")
print(f"{'Word':<20} {'BoW Count':<12} {'TF-IDF Score'}")
print("-" * 50)
for word, bow_count, tfidf_score in comparison[:15]:
    print(f"{word:<20} {int(bow_count):<12} {tfidf_score:.4f}")

### 💡 Exercise 2: Feature Analysis

1. Which words have high BoW counts but low TF-IDF scores? Why?
2. Which words have high TF-IDF scores? What makes them informative?
3. Try creating a vectorizer with different parameters (e.g., include bigrams with `ngram_range=(1,2)`)

In [ ]:
# Your experimental vectorizer here
# Example:
# custom_vectorizer = TfidfVectorizer(
#     ngram_range=(1, 2),  # Include unigrams and bigrams
#     max_features=10000,
#     ...
# )


## Part 3: Building Classifiers

Now let's train different types of classifiers and compare their performance.

### Classifier 1: Naive Bayes

Naive Bayes assumes features are independent and uses Bayes' theorem:

$$P(y|x) \propto P(y) \prod_{i=1}^n P(x_i|y)$$

Despite the "naive" independence assumption, it works well for text!

In [ ]:
# Train Naive Bayes with BoW features
print("Training Naive Bayes with BoW features...")
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, train_data.target)

# Predict on test set
y_pred_nb_bow = nb_bow.predict(X_test_bow)
acc_nb_bow = accuracy_score(test_data.target, y_pred_nb_bow)

print(f"✓ Accuracy: {acc_nb_bow:.4f}")
print("\nClassification Report:")
print(classification_report(test_data.target, y_pred_nb_bow, target_names=test_data.target_names))

In [ ]:
# Train Naive Bayes with TF-IDF features
print("Training Naive Bayes with TF-IDF features...")
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, train_data.target)

y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
acc_nb_tfidf = accuracy_score(test_data.target, y_pred_nb_tfidf)

print(f"✓ Accuracy: {acc_nb_tfidf:.4f}")
print("\nClassification Report:")
print(classification_report(test_data.target, y_pred_nb_tfidf, target_names=test_data.target_names))

### Classifier 2: Logistic Regression

Logistic Regression learns weights for each feature:

$$P(y=k|x) = \frac{e^{w_k^T x}}{\sum_{j} e^{w_j^T x}}$$

It's a linear model but often works better than Naive Bayes.

In [ ]:
# Train Logistic Regression with TF-IDF
print("Training Logistic Regression with TF-IDF features...")
lr = LogisticRegression(
    max_iter=1000,
    C=1.0,  # Regularization strength
    random_state=42
)
lr.fit(X_train_tfidf, train_data.target)

y_pred_lr = lr.predict(X_test_tfidf)
acc_lr = accuracy_score(test_data.target, y_pred_lr)

print(f"✓ Accuracy: {acc_lr:.4f}")
print("\nClassification Report:")
print(classification_report(test_data.target, y_pred_lr, target_names=test_data.target_names))

### Analyze Feature Weights

Logistic Regression gives us interpretable feature weights!

In [ ]:
def show_most_informative_features(vectorizer, classifier, n=10):
    """
    Show the most important features for each class in a linear classifier.
    """
    feature_names = vectorizer.get_feature_names_out()
    
    for i, category in enumerate(train_data.target_names):
        print(f"\n{'='*60}")
        print(f"Most informative features for: {category}")
        print(f"{'='*60}")
        
        # Get weights for this class
        weights = classifier.coef_[i]
        
        # Get top positive and negative features
        top_indices = np.argsort(weights)[-n:]
        bottom_indices = np.argsort(weights)[:n]
        
        print(f"\nTop {n} positive features (predicting {category}):")
        for idx in reversed(top_indices):
            print(f"  {feature_names[idx]:<20} {weights[idx]:>8.4f}")

show_most_informative_features(tfidf_vectorizer, lr, n=15)

### 💡 Exercise 3: Feature Interpretation

1. Do the top features for each class make sense?
2. Are there any surprising features with high weights?
3. How might you improve the feature set based on these results?

### Classifier 3: Simple Neural Network with PyTorch

Let's build a simple feedforward neural network for comparison.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert sparse matrices to dense tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray()).to(device)
y_train_tensor = torch.LongTensor(train_data.target).to(device)
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray()).to(device)
y_test_tensor = torch.LongTensor(test_data.target).to(device)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Created {len(train_loader)} batches for training")

In [ ]:
# Define a simple neural network
class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Initialize model
input_dim = X_train_tfidf.shape[1]
hidden_dim = 256
output_dim = len(train_data.target_names)

model = TextClassifier(input_dim, hidden_dim, output_dim).to(device)
print(model)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"\nModel has {sum(p.numel() for p in model.parameters())} parameters")

In [ ]:
# Training loop
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    """
    Train the neural network.
    """
    model.train()
    history = {'loss': [], 'accuracy': []}
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Statistics
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        avg_loss = total_loss / len(train_loader)
        accuracy = correct / total
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    return history

# Train the model
print("Training neural network...\n")
history = train_model(model, train_loader, criterion, optimizer, epochs=10)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'])
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(history['accuracy'])
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
    y_pred_nn = predicted.cpu().numpy()

acc_nn = accuracy_score(test_data.target, y_pred_nn)
print(f"Neural Network Test Accuracy: {acc_nn:.4f}")
print("\nClassification Report:")
print(classification_report(test_data.target, y_pred_nn, target_names=test_data.target_names))

## Part 4: Model Comparison and Analysis

In [ ]:
# Compare all models
results = pd.DataFrame({
    'Model': [
        'Naive Bayes (BoW)',
        'Naive Bayes (TF-IDF)',
        'Logistic Regression',
        'Neural Network'
    ],
    'Accuracy': [
        acc_nb_bow,
        acc_nb_tfidf,
        acc_lr,
        acc_nn
    ]
})

results = results.sort_values('Accuracy', ascending=False)
print(results.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 5))
plt.barh(results['Model'], results['Accuracy'])
plt.xlabel('Accuracy')
plt.title('Model Comparison')
plt.xlim([0.7, 1.0])
for i, v in enumerate(results['Accuracy']):
    plt.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model
best_predictions = y_pred_lr  # Use logistic regression

cm = confusion_matrix(test_data.target, best_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_data.target_names,
            yticklabels=test_data.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()

### 💡 Exercise 4: Error Analysis

Let's examine where our best model makes mistakes.

In [ ]:
# Find misclassified examples
def show_errors(true_labels, predictions, data, n=5):
    """
    Show misclassified examples.
    """
    errors = np.where(true_labels != predictions)[0]
    
    print(f"Found {len(errors)} misclassified documents\n")
    
    for i, idx in enumerate(errors[:n], 1):
        print(f"\n{'='*80}")
        print(f"Error {i}:")
        print(f"{'='*80}")
        print(f"True label: {data.target_names[true_labels[idx]]}")
        print(f"Predicted:  {data.target_names[predictions[idx]]}")
        print(f"\nText preview:")
        print(data.data[idx][:400].replace('\n', ' '))
        print("...")

show_errors(test_data.target, best_predictions, test_data, n=3)

**Questions:**
1. Why do you think these documents were misclassified?
2. What patterns do you see in the errors?
3. How could you improve the classifier to fix these errors?

---

## Part 5: Building Your Own Classifier

### 💡 Exercise 5: Improve the Classifier

Try these improvements:

1. **Better preprocessing**: Remove numbers, lemmatize, handle contractions
2. **Different features**: Try character n-grams, word n-grams (bigrams, trigrams)
3. **Feature selection**: Use chi-squared test or mutual information
4. **Hyperparameter tuning**: Adjust C in LogisticRegression, hidden_dim in neural net
5. **Ensemble methods**: Combine multiple classifiers

Can you beat the baseline results?

In [ ]:
# Your improved classifier here

# Example: Better preprocessing
def preprocess_text(text):
    """
    Custom preprocessing function.
    """
    # Lowercase
    text = text.lower()
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# TODO: Apply preprocessing and retrain
# preprocessed_train = [preprocess_text(doc) for doc in train_data.data]
# preprocessed_test = [preprocess_text(doc) for doc in test_data.data]

# TODO: Try different vectorizers
# improved_vectorizer = TfidfVectorizer(
#     ngram_range=(1, 3),  # Unigrams, bigrams, trigrams
#     max_features=10000,
#     ...
# )

# TODO: Train and evaluate


## Part 6: Real-World Considerations

### Class Imbalance

What if some classes have many more examples than others?

In [ ]:
# Simulate class imbalance
def create_imbalanced_dataset(X, y, imbalance_ratio=0.1):
    """
    Create an imbalanced dataset by downsampling some classes.
    """
    # Keep all examples of first class
    # Keep only imbalance_ratio of other classes
    indices_to_keep = []
    
    for class_id in range(len(train_data.target_names)):
        class_indices = np.where(y == class_id)[0]
        
        if class_id == 0:
            # Keep all examples of first class
            indices_to_keep.extend(class_indices)
        else:
            # Downsample other classes
            n_to_keep = int(len(class_indices) * imbalance_ratio)
            sampled = np.random.choice(class_indices, size=n_to_keep, replace=False)
            indices_to_keep.extend(sampled)
    
    return indices_to_keep

# Create imbalanced training set
imbalanced_indices = create_imbalanced_dataset(X_train_tfidf, train_data.target)
X_train_imb = X_train_tfidf[imbalanced_indices]
y_train_imb = train_data.target[imbalanced_indices]

print("Imbalanced class distribution:")
for i, name in enumerate(train_data.target_names):
    count = np.sum(y_train_imb == i)
    print(f"  {name}: {count}")

# Train on imbalanced data
lr_imb = LogisticRegression(max_iter=1000, random_state=42)
lr_imb.fit(X_train_imb, y_train_imb)

y_pred_imb = lr_imb.predict(X_test_tfidf)
print("\nResults on imbalanced training:")
print(classification_report(test_data.target, y_pred_imb, target_names=test_data.target_names))

In [ ]:
# Fix with class weights
lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Automatically adjust weights
    random_state=42
)
lr_balanced.fit(X_train_imb, y_train_imb)

y_pred_balanced = lr_balanced.predict(X_test_tfidf)
print("Results with balanced class weights:")
print(classification_report(test_data.target, y_pred_balanced, target_names=test_data.target_names))

## Part 7: Discussion Questions

Discuss with your group:

1. **BoW vs TF-IDF**: When would you prefer one over the other? What are the trade-offs?

2. **Linear vs Neural**: Why didn't the neural network significantly outperform logistic regression? When would you expect neural networks to do better?

3. **Feature Engineering**: How important was feature engineering compared to model choice? What does this tell us about NLP?

4. **Evaluation Metrics**: When would accuracy be a poor metric? What other metrics matter?

5. **Scalability**: Which approach would scale best to millions of documents? Thousands of classes?

6. **Interpretability**: Which models are most interpretable? Why does interpretability matter?

**Your notes:**

---

(Discussion notes here)

---

## Summary and Key Takeaways

Today you learned:

1. ✅ **Feature Engineering**: BoW, TF-IDF, and their trade-offs
2. ✅ **Multiple Approaches**: Naive Bayes, Logistic Regression, Neural Networks
3. ✅ **Evaluation**: Accuracy, precision, recall, F1, confusion matrices
4. ✅ **Debugging**: Error analysis and feature interpretation
5. ✅ **Real-World Issues**: Class imbalance, scalability, interpretability

### Key Insights:
- Good features matter more than complex models (for many tasks)
- TF-IDF usually beats raw BoW for text classification
- Linear models (LR) are competitive with neural networks on bag-of-words features
- Always examine errors to understand model behavior
- Consider class imbalance and adjust accordingly

### Next Week:
We'll explore **word embeddings** - dense, semantic representations that capture meaning!

---

**Questions? Office hours or Piazza!**